<a href="https://colab.research.google.com/github/irhamaam460-maker/KELOMPOK4_E-COMMERCE/blob/main/E_commerce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
ecommerce_semantic.py
Sistem E-Commerce berbasis Knowledge Base Semantik
Jalankan: python ecommerce_semantic.py
"""

# ============================================================
# KNOWLEDGE BASE DATA
# ============================================================

KB = {
    "entities": [
        {
            "id": "ecommerce",
            "name": "E-Commerce Platform",
            "type": "platform",
            "description": "Marketplace digital untuk produk elektronik.",
            "properties": {"domain": "Retail online", "skala": "Enterprise"},
        },
        {
            "id": "pengguna",
            "name": "Pengguna",
            "type": "aktor",
            "description": "Pembeli dan pencari produk.",
            "properties": {"peran": "Pembeli", "aktivitas": "Membeli & mencari produk"},
        },
        {
            "id": "elektronik",
            "name": "Elektronik",
            "type": "kategori",
            "description": "Kategori induk semua perangkat elektronik.",
            "properties": {"level": "Top-level", "sub_kategori": ["Laptop", "Smartphone", "Smart TV"]},
        },
        {
            "id": "laptop",
            "name": "Laptop",
            "type": "produk",
            "description": "Perangkat komputasi portabel untuk kerja dan produktivitas.",
            "properties": {"harga": 12000000, "stok": 50, "kategori": "Elektronik", "cocok_untuk": ["kerja", "kantor", "remote"]},
        },
        {
            "id": "smartphone",
            "name": "Smartphone",
            "type": "produk",
            "description": "Perangkat komunikasi mobile berbasis Android/iOS.",
            "properties": {"harga": 5000000, "stok": 120, "kategori": "Elektronik", "cocok_untuk": ["komunikasi", "hiburan"]},
        },
        {
            "id": "smart_tv",
            "name": "Smart TV",
            "type": "produk",
            "description": "Televisi pintar dengan fitur streaming dan internet.",
            "properties": {"harga": 8000000, "stok": 30, "kategori": "Elektronik", "cocok_untuk": ["hiburan", "rumah"]},
        },
        {
            "id": "kerja",
            "name": "Kerja",
            "type": "kebutuhan",
            "description": "Konteks kebutuhan profesional dan produktivitas.",
            "properties": {"intent": "Produktivitas", "sub_konteks": ["kantor", "remote"]},
        },
        {
            "id": "kantor",
            "name": "Kantor",
            "type": "konteks",
            "description": "Lingkungan kerja stasioner di kantor.",
            "properties": {"tipe": "Stasioner", "induk": "kerja"},
        },
        {
            "id": "remote",
            "name": "Remote / WFH",
            "type": "konteks",
            "description": "Bekerja dari rumah atau luar kantor.",
            "properties": {"tipe": "Fleksibel", "induk": "kerja"},
        },
    ],
    "relations": [
        {"from": "laptop",     "label": "kategori",         "to": "elektronik"},
        {"from": "smartphone", "label": "kategori",         "to": "elektronik"},
        {"from": "smart_tv",   "label": "kategori",         "to": "elektronik"},
        {"from": "laptop",     "label": "sesuai kebutuhan", "to": "kerja"},
        {"from": "laptop",     "label": "cocok untuk",      "to": "kantor"},
        {"from": "laptop",     "label": "cocok untuk",      "to": "remote"},
        {"from": "pengguna",   "label": "membeli",          "to": "laptop"},
        {"from": "pengguna",   "label": "membeli",          "to": "smartphone"},
        {"from": "pengguna",   "label": "membeli",          "to": "smart_tv"},
        {"from": "pengguna",   "label": "mencari berdasar", "to": "kerja"},
        {"from": "kerja",      "label": "memiliki konteks", "to": "kantor"},
        {"from": "kerja",      "label": "memiliki konteks", "to": "remote"},
    ],
    "keranjang": [],
    "riwayat_pembelian": [],
}


# ============================================================
# HELPER
# ============================================================

def get_entity(entity_id: str) -> dict | None:
    for e in KB["entities"]:
        if e["id"] == entity_id:
            return e
    return None

def get_products() -> list:
    return [e for e in KB["entities"] if e["type"] == "produk"]

def get_relations_from(entity_id: str) -> list:
    return [r for r in KB["relations"] if r["from"] == entity_id]

def format_harga(harga: int) -> str:
    return f"Rp {harga:,.0f}".replace(",", ".")

def separator(char="─", width=55):
    print(char * width)

def header(title: str):
    separator("═")
    print(f"  {title}")
    separator("═")


# ============================================================
# FITUR: LIHAT SEMUA PRODUK
# ============================================================

def lihat_produk():
    header("DAFTAR PRODUK")
    produk_list = get_products()
    for i, p in enumerate(produk_list, 1):
        stok = p["properties"]["stok"]
        harga = p["properties"]["harga"]
        status = "✓ Tersedia" if stok > 0 else "✗ Habis"
        print(f"  {i}. {p['name']}")
        print(f"     Harga : {format_harga(harga)}")
        print(f"     Stok  : {stok} unit  |  {status}")
        print(f"     Info  : {p['description']}")
        separator("·")


# ============================================================
# FITUR: REKOMENDASI SEMANTIK
# ============================================================

def rekomendasi():
    header("REKOMENDASI PRODUK")
    print("  Masukkan kebutuhanmu dan sistem akan merekomendasikan")
    print("  produk yang paling sesuai secara semantik.\n")

    kebutuhan_input = input("  Kebutuhanmu (contoh: kerja, hiburan, komunikasi): ").strip().lower()

    cocok = []
    for produk in get_products():
        cocok_untuk = produk["properties"].get("cocok_untuk", [])
        # Cek kecocokan semantik: langsung atau via relasi konteks
        if any(kebutuhan_input in c for c in cocok_untuk):
            cocok.append(produk)
        elif kebutuhan_input in produk["description"].lower():
            cocok.append(produk)
        elif kebutuhan_input in produk["name"].lower():
            cocok.append(produk)

    # Cek via relasi kebutuhan (misal input "kantor" → cari produk yang relasi ke kantor)
    if not cocok:
        for r in KB["relations"]:
            target = get_entity(r["to"])
            if target and kebutuhan_input in target["name"].lower():
                sumber = get_entity(r["from"])
                if sumber and sumber["type"] == "produk" and sumber not in cocok:
                    cocok.append(sumber)

    separator()
    if cocok:
        print(f"  Produk yang cocok untuk '{kebutuhan_input}':\n")
        for p in cocok:
            print(f"  ► {p['name']} — {format_harga(p['properties']['harga'])}")
            print(f"    {p['description']}")
    else:
        print(f"  Tidak ada produk yang cocok untuk '{kebutuhan_input}'.")
        print("  Coba kata kunci: kerja, kantor, remote, hiburan, komunikasi")
    separator()


# ============================================================
# FITUR: CARI PRODUK
# ============================================================

def cari_produk():
    header("CARI PRODUK")
    keyword = input("  Masukkan nama produk: ").strip().lower()

    hasil = [
        e for e in KB["entities"]
        if keyword in e["name"].lower() or keyword in e["description"].lower()
    ]

    separator()
    if hasil:
        for e in hasil:
            tipe = e["type"].upper()
            print(f"  [{tipe}] {e['name']}")
            print(f"    {e['description']}")
            if e["type"] == "produk":
                print(f"    Harga : {format_harga(e['properties']['harga'])}")
                print(f"    Stok  : {e['properties']['stok']} unit")
            separator("·")
    else:
        print(f"  Produk '{keyword}' tidak ditemukan.")
    separator()


# ============================================================
# FITUR: TAMBAH KE KERANJANG
# ============================================================

def tambah_keranjang():
    header("TAMBAH KE KERANJANG")
    produk_list = get_products()

    for i, p in enumerate(produk_list, 1):
        print(f"  {i}. {p['name']} — {format_harga(p['properties']['harga'])} (stok: {p['properties']['stok']})")

    separator()
    try:
        pilihan = int(input("  Pilih nomor produk (0 = batal): "))
        if pilihan == 0:
            return
        if not 1 <= pilihan <= len(produk_list):
            print("  Pilihan tidak valid.")
            return

        produk = produk_list[pilihan - 1]
        if produk["properties"]["stok"] == 0:
            print(f"  Stok {produk['name']} habis!")
            return

        jumlah = int(input(f"  Jumlah {produk['name']}: "))
        if jumlah <= 0:
            print("  Jumlah tidak valid.")
            return
        if jumlah > produk["properties"]["stok"]:
            print(f"  Stok tidak cukup. Stok tersedia: {produk['properties']['stok']}")
            return

        # Cek apakah sudah ada di keranjang
        for item in KB["keranjang"]:
            if item["id"] == produk["id"]:
                item["jumlah"] += jumlah
                print(f"\n  ✓ {produk['name']} x{jumlah} ditambahkan ke keranjang.")
                return

        KB["keranjang"].append({"id": produk["id"], "name": produk["name"], "harga": produk["properties"]["harga"], "jumlah": jumlah})
        print(f"\n  ✓ {produk['name']} x{jumlah} ditambahkan ke keranjang.")

    except ValueError:
        print("  Input tidak valid.")


# ============================================================
# FITUR: LIHAT KERANJANG
# ============================================================

def lihat_keranjang():
    header("KERANJANG BELANJA")
    if not KB["keranjang"]:
        print("  Keranjang kosong.")
        return

    total = 0
    for i, item in enumerate(KB["keranjang"], 1):
        subtotal = item["harga"] * item["jumlah"]
        total += subtotal
        print(f"  {i}. {item['name']}")
        print(f"     {format_harga(item['harga'])} x {item['jumlah']} = {format_harga(subtotal)}")
        separator("·")

    separator()
    print(f"  TOTAL: {format_harga(total)}")
    separator()


# ============================================================
# FITUR: CHECKOUT
# ============================================================

def checkout():
    header("CHECKOUT")
    if not KB["keranjang"]:
        print("  Keranjang kosong. Tambahkan produk terlebih dahulu.")
        return

    total = sum(item["harga"] * item["jumlah"] for item in KB["keranjang"])

    print("  Ringkasan pesanan:\n")
    for item in KB["keranjang"]:
        print(f"  • {item['name']} x{item['jumlah']} = {format_harga(item['harga'] * item['jumlah'])}")

    separator()
    print(f"  Total pembayaran: {format_harga(total)}")
    separator()

    konfirmasi = input("  Konfirmasi pembelian? (y/n): ").strip().lower()
    if konfirmasi != "y":
        print("  Checkout dibatalkan.")
        return

    # Kurangi stok
    for item in KB["keranjang"]:
        produk = get_entity(item["id"])
        if produk:
            produk["properties"]["stok"] -= item["jumlah"]

    # Simpan ke riwayat
    KB["riwayat_pembelian"].append({
        "items": KB["keranjang"].copy(),
        "total": total
    })

    KB["keranjang"].clear()
    print(f"\n  ✓ Pembelian berhasil! Total: {format_harga(total)}")
    print("  Terima kasih telah berbelanja di E-Commerce Platform.")
    separator()


# ============================================================
# FITUR: RIWAYAT PEMBELIAN
# ============================================================

def riwayat():
    header("RIWAYAT PEMBELIAN")
    if not KB["riwayat_pembelian"]:
        print("  Belum ada riwayat pembelian.")
        return

    for i, transaksi in enumerate(KB["riwayat_pembelian"], 1):
        print(f"  Transaksi #{i}")
        for item in transaksi["items"]:
            print(f"    • {item['name']} x{item['jumlah']} = {format_harga(item['harga'] * item['jumlah'])}")
        print(f"    Total: {format_harga(transaksi['total'])}")
        separator("·")


# ============================================================
# FITUR: RELASI SEMANTIK
# ============================================================

def lihat_relasi():
    header("RELASI SEMANTIK ENTITAS")
    print("  Pilih entitas:\n")

    entities = KB["entities"]
    for i, e in enumerate(entities, 1):
        print(f"  {i}. [{e['type'].upper()}] {e['name']}")

    separator()
    try:
        pilihan = int(input("  Pilih nomor (0 = batal): "))
        if pilihan == 0:
            return
        if not 1 <= pilihan <= len(entities):
            print("  Pilihan tidak valid.")
            return

        entity = entities[pilihan - 1]
        rels = get_relations_from(entity["id"])

        separator()
        print(f"  Entitas  : {entity['name']} [{entity['type'].upper()}]")
        print(f"  Deskripsi: {entity['description']}")
        separator("·")
        if rels:
            print("  Relasi semantik:\n")
            for r in rels:
                target = get_entity(r["to"])
                tname = target["name"] if target else r["to"]
                print(f"    → [{r['label']}] {tname}")
        else:
            print("  Tidak ada relasi keluar dari entitas ini.")
        separator()

    except ValueError:
        print("  Input tidak valid.")


# ============================================================
# MAIN MENU
# ============================================================

def main():
    menu = {
        "1": ("Lihat Semua Produk",       lihat_produk),
        "2": ("Cari Produk",              cari_produk),
        "3": ("Rekomendasi Semantik",     rekomendasi),
        "4": ("Tambah ke Keranjang",      tambah_keranjang),
        "5": ("Lihat Keranjang",          lihat_keranjang),
        "6": ("Checkout",                 checkout),
        "7": ("Riwayat Pembelian",        riwayat),
        "8": ("Lihat Relasi Semantik",    lihat_relasi),
        "0": ("Keluar",                   None),
    }

    print("\n" + "═" * 55)
    print("  🛒  E-COMMERCE PLATFORM — SEMANTIC KNOWLEDGE BASE")
    print("═" * 55)

    while True:
        print("\n  MENU UTAMA\n")
        for key, (label, _) in menu.items():
            print(f"  [{key}] {label}")

        separator()
        pilihan = input("  Pilih menu: ").strip()

        if pilihan == "0":
            print("\n  Sampai jumpa! 👋\n")
            break
        elif pilihan in menu:
            print()
            menu[pilihan][1]()
        else:
            print("  Menu tidak tersedia. Coba lagi.")


if __name__ == "__main__":
    main()


═══════════════════════════════════════════════════════
  🛒  E-COMMERCE PLATFORM — SEMANTIC KNOWLEDGE BASE
═══════════════════════════════════════════════════════

  MENU UTAMA

  [1] Lihat Semua Produk
  [2] Cari Produk
  [3] Rekomendasi Semantik
  [4] Tambah ke Keranjang
  [5] Lihat Keranjang
  [6] Checkout
  [7] Riwayat Pembelian
  [8] Lihat Relasi Semantik
  [0] Keluar
───────────────────────────────────────────────────────
  Pilih menu: 1

═══════════════════════════════════════════════════════
  DAFTAR PRODUK
═══════════════════════════════════════════════════════
  1. Laptop
     Harga : Rp 12.000.000
     Stok  : 50 unit  |  ✓ Tersedia
     Info  : Perangkat komputasi portabel untuk kerja dan produktivitas.
·······················································
  2. Smartphone
     Harga : Rp 5.000.000
     Stok  : 120 unit  |  ✓ Tersedia
     Info  : Perangkat komunikasi mobile berbasis Android/iOS.
·······················································
  3. Smart TV
     H